# 01 — Pipeline smoke test on 5 known CLAGN

**Milestone 2 (critical early gate).** Before scaling to 10^5–10^6 objects, confirm that Fornax's `light_curve_collector` recovers the *known* mid-IR (NEOWISE W1/W2) and optical (ZTF g) changing-look events for five well-documented CLAGN.

**Pass criterion:** for each target, the recovered light curve shows the expected event (see `expected_event` in `src/smoke_test_targets.py`) at roughly the published amplitude. If the **mid-IR-selected** slot-5 object is missed but the bright archetypes pass, that flags a sensitivity limit in our discovery channel — informative either way.

> Run this **on the Fornax Science Console** (default `python3` kernel) where the `light_curve_collector` helper modules and archive access live.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord

from src.smoke_test_targets import TARGETS
from src import selection

for t in TARGETS:
    print(f"{t.name:28s} {t.direction:9s} {t.expected_event}")

## 1. Build the source table (resolve coordinates by name)

Resolve each target via SIMBAD/NED rather than trusting the cached fallback coords. Skip the slot-5 TODO object until it's pinned.

In [ ]:
rows = []
for i, t in enumerate(TARGETS):
    if t.name.startswith('TODO'):
        print(f"skipping unpinned slot: {t.name}")
        continue
    try:
        c = SkyCoord.from_name(t.name)
        ra, dec = c.ra.deg, c.dec.deg
    except Exception as e:
        print(f"name resolution failed for {t.name} ({e}); using fallback coords")
        ra, dec = t.ra_deg, t.dec_deg
    rows.append({'objectid': i, 'label': t.name, 'ra': ra, 'dec': dec,
                 'direction': t.direction})

sample = pd.DataFrame(rows)
sample

## 2. Collect multi-wavelength light curves

**TODO — wire to the actual `light_curve_collector` API.** Confirm the exact import/call signature against the notebook in `nasa-fornax/fornax-demo-notebooks` (`light_curves/light_curve_collector.md`). The repo ships helper modules (e.g. a `MultiIndexDFObject` data structure and per-archive query functions for ZTF and WISE/NEOWISE). Typical shape:

```python
from data_structures import MultiIndexDFObject
from ztf_functions import ZTF_get_lightcurve
from wise_functions import WISE_get_lightcurves

df_lc = MultiIndexDFObject()
df_lc.append(ZTF_get_lightcurve(sample, ...))
df_lc.append(WISE_get_lightcurves(sample, radius=1.0, bandlist=['W1','W2']))
```

Replace the stub below once the signatures are confirmed on the Console.

In [ ]:
# STUB — replace with the real light_curve_collector calls on Fornax.
# Expected result: a long-format table with columns like
#   [objectid, label, band, time(MJD), flux_or_mag, err]
# covering bands W1, W2, ZTF_g (and ZTF_r where available).

raise NotImplementedError(
    'Wire this cell to light_curve_collector on the Fornax Console, then '
    'produce a dataframe `lc` with columns [objectid, label, band, mjd, mag, magerr].'
)

# lc = ...  # noqa

## 3. Plot each target (W1, W2, ZTF g) and eyeball the event

In [ ]:
def plot_target(lc, label):
    sub = lc[lc['label'] == label]
    fig, ax = plt.subplots(figsize=(9, 4))
    for band, marker in [('W1', 'o'), ('W2', 's'), ('ZTF_g', '.')]:
        b = sub[sub['band'] == band].sort_values('mjd')
        if len(b):
            ax.errorbar(b['mjd'], b['mag'], yerr=b['magerr'], fmt=marker,
                        ms=4, alpha=0.7, label=band)
    ax.invert_yaxis()
    ax.set_xlabel('MJD'); ax.set_ylabel('mag'); ax.set_title(label)
    ax.legend()
    return fig

# for label in sample['label']:
#     plot_target(lc, label)
# plt.show()

## 4. Quantitative check vs. literature thresholds

Compute the stage-3 metrics per target and confirm each known CLAGN clears the prior thresholds in `src/selection.py`. This is a sanity check on the metrics, not the production selection (which is injection-recovery-calibrated).

In [ ]:
# def features_for(lc, label):
#     sub = lc[lc['label'] == label]
#     w1 = sub[sub['band'] == 'W1'].sort_values('mjd')
#     w2 = sub[sub['band'] == 'W2'].sort_values('mjd')
#     g  = sub[sub['band'] == 'ZTF_g']
#     return {
#         'delta_w1': selection.delta_mag(w1['mag']),
#         'delta_w2': selection.delta_mag(w2['mag']),
#         'delta_g':  selection.delta_mag(g['mag']),
#         'wise_color_change': selection.wise_color_change(w1['mag'].values, w2['mag'].values),
#         'trend_w1': selection.monotonic_trend(w1['mjd'], w1['mag']),
#     }
#
# results = {lab: features_for(lc, lab) for lab in sample['label']}
# res_df = pd.DataFrame(results).T
# res_df['is_candidate'] = [selection.is_candidate(r) for r in results.values()]
# res_df

## 5. Verdict

- [ ] All 4 archetypes recovered with expected events
- [ ] Mid-IR-selected slot-5 object recovered (once pinned)
- [ ] Metrics clear prior thresholds for all turn-on AND turn-off cases
- [ ] Runtime per object recorded → extrapolate compute-credit budget for the full sample

If yes → proceed to parent-sample assembly + `scale_up`. If the pipeline misses known events → redesign before scaling.